In [173]:
from io import StringIO

import pandas as pd
import requests

def requestConstituents(url, table_id):
    response = requests.get(
        url,
        headers={"User-Agent": "AAIS student project"},
        timeout=30,
    )
    response.raise_for_status()

    tables = pd.read_html(
        StringIO(response.text),
        attrs={"id": table_id},
        flavor="lxml",
    )
    df = tables[0]
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            "_".join(dict.fromkeys(str(level) for level in column if level))
            for column in df.columns
        ]
    df.columns = df.columns.str.lower().str.replace(" ", "_")
 
    
    return df

constituents_df = (
    requestConstituents(
        "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
        "constituents",
    )
)
history_df = (
    requestConstituents(
        "https://en.wikipedia.org/wiki/Historical_components_of_the_S&P_500",
          "changes",
          )
)
constituents_df["yahoo_symbol"] = constituents_df["symbol"].str.replace(".", "-", regex=False)

history_df['yahoo_added_ticker'] = history_df['added_ticker'].str.replace(".", "-", regex=False)
history_df['yahoo_removed_ticker'] = history_df['removed_ticker'].str.replace(".", "-", regex=False)


In [174]:
def prepare_events_df(history_df):
    df = history_df.copy()
    df["date"] = pd.to_datetime(df["effective_date"], format="%B %d, %Y", errors="coerce")

    added = df[["date", "added_ticker"]].rename(columns={"added_ticker": "ticker"}).assign(action="add")
    removed = df[["date", "removed_ticker"]].rename(columns={"removed_ticker": "ticker"}).assign(action="remove")

    events_df = pd.concat([added, removed], ignore_index=True)
    events_df["ticker"] = events_df["ticker"].astype("string").str.strip()
    events_df = events_df.dropna(subset=["date", "ticker"])
    events_df = events_df[events_df["ticker"] != ""]

    return events_df.sort_values("date").reset_index(drop=True)



def members_on(intervals, date):
    d = pd.Timestamp(date)
    started = intervals["start"].isna() | (intervals["start"] <= d)
    not_ended = intervals["end"].isna() | (intervals["end"] > d)
    return sorted(intervals.loc[started & not_ended, "ticker"])



In [175]:
events_df = prepare_events_df(history_df)
intervals = []

for event in events_df.sort_values("date").itertuples():
    if event.action == "add":
        intervals.append((event.ticker, event.date, None))
    else:
        for i in range(len(intervals)):
            if intervals[i][0] == event.ticker and intervals[i][2] is None:
                intervals[i] = (intervals[i][0], intervals[i][1], event.date)
                break
        else:
            intervals.append((event.ticker, None, event.date))

constituents_df["date_added"] = pd.to_datetime(constituents_df["date_added"], errors="coerce")

open_tickers = {t for t, _, end in intervals if end is None}
for row in constituents_df.itertuples():
    if row.yahoo_symbol not in open_tickers:
        intervals.append((row.yahoo_symbol, row.date_added, None))

In [ ]:
iv = pd.DataFrame(intervals, columns=["ticker", "start", "end"])

current = set(constituents_df["yahoo_symbol"])
open_tickers = set(iv.loc[iv["end"].isna(), "ticker"])



phantoms = sorted(open_tickers - current)


iv = iv[~iv["ticker"].isin(phantoms)]

iv = iv.reset_index(drop=True)

for d in ["2012-01-03", "2016-01-04", "2020-01-02", "2024-01-02"]:
    print(d, len(members_on(iv, d)))

2012-01-03 497
2016-01-04 501
2020-01-02 504
2024-01-02 503


In [177]:
data = members_on(iv, "2016-01-02")

In [178]:
data

['A',
 'AA',
 'AAL',
 'AAP',
 'AAPL',
 'ABBV',
 'ABT',
 'ACE',
 'ACN',
 'ADBE',
 'ADI',
 'ADM',
 'ADP',
 'ADS',
 'ADSK',
 'ADT',
 'AEE',
 'AEP',
 'AES',
 'AET',
 'AFL',
 'AGN',
 'AIG',
 'AIV',
 'AIZ',
 'AKAM',
 'ALL',
 'ALLE',
 'ALXN',
 'AMAT',
 'AME',
 'AMG',
 'AMGN',
 'AMP',
 'AMT',
 'AMZN',
 'AN',
 'ANDV',
 'AON',
 'APA',
 'APC',
 'APD',
 'APH',
 'APTV',
 'ARG',
 'ATVI',
 'AVB',
 'AVGO',
 'AVY',
 'AXP',
 'AZO',
 'BA',
 'BAC',
 'BALL',
 'BAX',
 'BBBY',
 'BBWI',
 'BBY',
 'BCR',
 'BDX',
 'BEN',
 'BF-B',
 'BHI',
 'BIIB',
 'BKNG',
 'BLK',
 'BMY',
 'BNY',
 'BRCM',
 'BRK-B',
 'BSX',
 'BWA',
 'BXLT',
 'BXP',
 'C',
 'CA',
 'CAG',
 'CAH',
 'CAM',
 'CAT',
 'CB',
 'CBRE',
 'CCE',
 'CCI',
 'CCL',
 'CELG',
 'CERN',
 'CF',
 'CHD',
 'CHK',
 'CHRW',
 'CI',
 'CINF',
 'CL',
 'CLX',
 'CMA',
 'CMCSA',
 'CME',
 'CMG',
 'CMI',
 'CMS',
 'CNP',
 'CNX',
 'COF',
 'COL',
 'COP',
 'COR',
 'COST',
 'CPB',
 'CPGX',
 'CPRI',
 'CRM',
 'CSCO',
 'CSRA',
 'CSX',
 'CTAS',
 'CTRA',
 'CTSH',
 'CTXS',
 'CVC',
 'CVS',
 'CV